# 研发费用加计扣除政策、研发投入调整与企业创新效率
## ——聚焦分析 Notebook

**研究问题**: 2021年制造业研发费用加计扣除比例从75%提高至100%，是否伴随企业研发投入调整和相对创新效率改善？

**核心逻辑**:
- 创新数量DID不显著 → 研究转向
- 制造业研发投入相对增长较慢
- 单位研发投入和单位研发人员创新效率相对提升
- DID(效率) ≈ DID(创新产出) − DID(研发投入)

**数据**: A股上市公司 2017-2022年面板，CSMAR 8张表合并

**模型**: 双重固定效应 (企业FE + 年份FE)，企业层面聚类标准误

## 0. 环境配置与导入

In [ ]:
import pandas as pd
import numpy as np
from linearmodels import PanelOLS
import statsmodels.api as sm
from scipy import stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# 输出目录
OUTPUT_DIR = 'outputs'
FIGURE_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# Matplotlib 配置
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 中文字体
try:
    matplotlib.rcParams['font.family'] = 'Noto Sans CJK SC'
except:
    try:
        matplotlib.rcParams['font.family'] = 'WenQuanYi Micro Hei'
    except:
        pass

matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 300
matplotlib.rcParams['savefig.dpi'] = 300
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['axes.facecolor'] = 'white'
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.titlesize'] = 12
matplotlib.rcParams['axes.labelsize'] = 10

# ============================================================
# 学术配色方案 — 低饱和度，适合论文和PPT
# ============================================================
C_BLUE      = '#4472C4'   # 柔和蓝 — 制造业
C_ORANGE    = '#ED7D31'   # 暖橙 — 非制造业
C_RED       = '#C0504D'   # 砖红 — 政策线/假想时点
C_GREEN     = '#548235'   # 森林绿 — 效率/显著正
C_PURPLE    = '#8064A2'   # 柔和紫 — 强固定效应
C_TEAL      = '#5B9BD5'   # 钢青 — 补充色
C_GOLD      = '#BF8F00'   # 暗金 — 标注用
C_GRAY      = '#A5A5A5'   # 中性灰 — 不显著
C_DARK      = '#404040'   # 深灰 — 文字

# 语义映射
C_MFG       = C_BLUE
C_NONMFG    = C_ORANGE
C_POLICY    = C_RED
C_EFF       = C_GREEN
C_SIG_POS   = C_GREEN
C_SIG_NEG   = C_RED
C_NS        = C_GRAY
C_STRONG_FE = C_PURPLE
C_HATCH1    = '///'
C_HATCH2    = '\\\\\\'

print(f'开始时间: {datetime.now()}')
print(f'输出目录: {OUTPUT_DIR}')


## 工具函数

In [ ]:
def winsorize(s, lo=0.01, hi=0.99):
    """对 pandas Series 进行缩尾处理"""
    s = s.copy()
    vlo, vhi = s.quantile(lo), s.quantile(hi)
    return s.clip(vlo, vhi)


def run_panel_did(df, dep_var, exog_vars, entity_effects=True, time_effects=True,
                  cluster_entity=True, weights=None):
    """运行 PanelOLS DID 模型: 企业FE + 年份FE + 企业层面聚类标准误"""
    all_vars = exog_vars + [dep_var]
    sub = df.dropna(subset=all_vars).copy()
    if len(sub) == 0:
        return {'N': 0, 'error': 'no observations'}
    sub = sub.set_index(['stock_code', 'year'])
    try:
        mod = PanelOLS(
            dependent=sub[dep_var],
            exog=sub[exog_vars],
            entity_effects=entity_effects,
            time_effects=time_effects,
            weights=weights,
            drop_absorbed=True,
        )
        res = mod.fit(cov_type='clustered', cluster_entity=cluster_entity)
        out = {
            'N': len(sub),
            'firms': sub.index.get_level_values(0).nunique(),
            'r2_within': res.rsquared_within,
            'r2_overall': res.rsquared,
        }
        for v in exog_vars:
            if v in res.params.index:
                out[f'{v}_coef'] = res.params[v]
                out[f'{v}_se'] = res.std_errors[v]
                out[f'{v}_p'] = res.pvalues[v]
            else:
                out[f'{v}_coef'] = np.nan
                out[f'{v}_se'] = np.nan
                out[f'{v}_p'] = np.nan
        return out
    except Exception as e:
        return {'N': len(sub), 'error': str(e)[:120]}


def fmt_sig(p):
    """显著性星号标记"""
    if pd.isna(p):
        return ''
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.1:
        return '*'
    return ''


def save_fig(fig, name):
    """保存为 PNG + PDF (300dpi)"""
    fig.savefig(os.path.join(FIGURE_DIR, f'{name}.png'), dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig.savefig(os.path.join(FIGURE_DIR, f'{name}.pdf'), bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f'  Saved: {name}.png / {name}.pdf')


print('工具函数定义完成。')

## 1. 加载数据

In [ ]:
v4 = pd.read_csv('data/firm_panel_v4.csv')

print(f'V4面板: {len(v4):,} obs, {v4["stock_code"].nunique()} firms')
print(f'年份范围: {sorted(v4["year"].unique())}')
print(f'\n变量列表 ({len(v4.columns)}):')
print(v4.columns.tolist())

## 2. 数据审计

In [ ]:
# 基准样本: 2017-2022
bench = v4[(v4['year'] >= 2017) & (v4['year'] <= 2022)].copy()
print(f'2017-2022 基准样本: {len(bench):,} obs, {bench["stock_code"].nunique()} firms')

# 面板唯一性检查
dup_check = bench.groupby(['stock_code', 'year']).size()
n_dup = (dup_check > 1).sum()
print(f'stock_code-year 重复: {n_dup}')

# 每企业观测数
max_obs = bench.groupby('stock_code').size()
print(f'每企业最多观测: {max_obs.max()} 条')
print(f'恰好6条企业: {(max_obs == 6).sum()}/{len(max_obs)} ({(max_obs == 6).mean():.1%})')

# 扩展样本
ext = v4[(v4['year'] >= 2017) & (v4['year'] <= 2024)].copy()
max_obs_ext = ext.groupby('stock_code').size()
print(f'\n2017-2024 扩展: 每企业最多 {max_obs_ext.max()} 条')

In [ ]:
# 核心变量缺失率
audit_vars = [
    'stock_code', 'year',
    'invention_apply', 'invention_grant', 'patent_apply_total', 'patent_grant_total',
    'rd_expense', 'rd_intensity', 'rd_intensity_raw', 'rd_staff', 'rd_staff_ratio',
    'ln_assets', 'roa', 'cashflow_ratio', 'firm_age',
    'manufacturing', 'soe', 'industry_code', 'province_clean',
    'pre_rd_intensity', 'policy_exposure',
    'province_sci_tech_ratio',
]

audit_rows = []
for v in audit_vars:
    if v in bench.columns:
        missing = bench[v].isna().mean()
        n_miss = bench[v].isna().sum()
        audit_rows.append({'variable': v, 'missing_rate': missing, 'n_missing': n_miss, 'n_total': len(bench)})
    else:
        audit_rows.append({'variable': v, 'missing_rate': np.nan, 'n_missing': -1, 'n_total': len(bench)})

audit_df = pd.DataFrame(audit_rows)
print('核心变量缺失率:')
display(audit_df)

In [ ]:
# 金额变量单位检查
unit_checks = {}
for v in ['rd_expense', 'total_assets', 'revenue']:
    if v in bench.columns:
        s = bench[v].dropna()
        unit_checks[v] = {'mean': s.mean(), 'median': s.median()}
        print(f'{v:20s}: mean={s.mean():,.0f}, median={s.median():,.0f} yuan')

# 比率变量口径检查
print('\n比率变量口径:')
for v in ['rd_intensity', 'rd_intensity_raw', 'rd_staff_ratio', 'roa', 'cashflow_ratio']:
    if v in bench.columns:
        s = bench[v].dropna()
        scale = '0-1' if s.max() <= 1.05 else ('百分比' if s.max() > 10 else '?')
        print(f'  {v:25s}: mean={s.mean():.4f}, range=[{s.min():.4f}, {s.max():.4f}] → {scale}')

## 3. 变量构造

In [ ]:
# ---------- 比率变量百分比→0-1 ----------
bench['rd_intensity_01'] = bench['rd_intensity'] / 100
bench['rd_staff_ratio_01'] = bench['rd_staff_ratio'] / 100

# ---------- 研发支出万元对数 ----------
bench['ln_rd_expense_10k'] = np.log1p(bench['rd_expense'] / 10000)

# ---------- 阶段变量 ----------
bench['post2021'] = (bench['year'] >= 2021).astype(int)
bench['post2023'] = (bench['year'] >= 2023).astype(int)
bench['treat_2021_2022'] = bench['manufacturing'] * ((bench['year'] >= 2021) & (bench['year'] <= 2022)).astype(int)
bench['treat_2023_2024'] = bench['manufacturing'] * (bench['year'] >= 2023).astype(int)

# ---------- 创新效率: 对数差分 ----------
bench['eff_apply_rd_10k'] = bench['ln_invention_apply'] - bench['ln_rd_expense_10k']
bench['eff_grant_rd_10k'] = bench['ln_invention_grant'] - bench['ln_rd_expense_10k']
bench['eff_apply_staff'] = bench['ln_invention_apply'] - bench['ln_rd_staff']
bench['eff_grant_staff'] = bench['ln_invention_grant'] - bench['ln_rd_staff']

# ---------- 比率型效率变量 ----------
for num, den, name in [
    ('invention_apply', 'rd_expense', 'apply_per_rd'),
    ('invention_grant', 'rd_expense', 'grant_per_rd'),
    ('invention_apply', 'rd_staff', 'apply_per_staff'),
    ('invention_grant', 'rd_staff', 'grant_per_staff'),
]:
    denom = bench[den]
    bench[name] = np.where(denom > 0, bench[num] / denom, np.nan)

# ---------- 比率型变量1%/99%缩尾 + asinh ----------
for name in ['apply_per_rd', 'grant_per_rd', 'apply_per_staff', 'grant_per_staff']:
    bench[name + '_w'] = winsorize(bench[name], 0.01, 0.99)
    bench['asinh_' + name] = np.arcsinh(bench[name + '_w'])

# ---------- 政策前研发基础 ----------
pre_rd = bench[bench['year'].between(2017, 2020)].groupby('stock_code')['rd_intensity_01'].mean()
pre_rd.name = 'pre_rd_intensity_01'
bench = bench.merge(pre_rd, on='stock_code', how='left')

median_pre_rd = bench['pre_rd_intensity_01'].median()
q75_pre_rd = bench['pre_rd_intensity_01'].quantile(0.75)
bench['high_pre_rd'] = (bench['pre_rd_intensity_01'] > median_pre_rd).astype(int)
bench['high_pre_rd_q4'] = (bench['pre_rd_intensity_01'] > q75_pre_rd).astype(int)

# ---------- 政策暴露强度 ----------
bench['policy_exposure_new'] = bench['pre_rd_intensity_01'] * bench['manufacturing_post2021']

# ---------- 非国有企业 ----------
bench['private'] = 1 - bench['soe']

print(f'变量构造完成。样本: {len(bench):,} obs')
print(f'\n新构造变量:')
new_vars = ['rd_intensity_01', 'rd_staff_ratio_01', 'ln_rd_expense_10k',
            'eff_apply_rd_10k', 'eff_grant_rd_10k', 'eff_apply_staff', 'eff_grant_staff',
            'high_pre_rd', 'high_pre_rd_q4', 'pre_rd_intensity_01', 'private']
for v in new_vars:
    if v in bench.columns:
        s = bench[v].dropna()
        print(f'  {v:30s}: N={len(s):,}, mean={s.mean():.4f}, std={s.std():.4f}')

## 控制变量

基准控制变量: `ln_assets`, `roa`, `cashflow_ratio`, `firm_age`

注: `lev` (资产负债率) 不可用 (CSMAR资产负债表缺少负债数据)。使用 `cashflow_ratio` 作为补充控制。

In [ ]:
BASE_CONTROLS = ['ln_assets', 'roa', 'cashflow_ratio', 'firm_age']
print(f'基准控制变量: {BASE_CONTROLS}')

## 4. 创新数量基准模型

In [ ]:
quantity_deps = ['ln_invention_apply', 'ln_invention_grant',
                 'ln_patent_apply_total', 'ln_patent_grant_total']
quantity_results = []

for dep in quantity_deps:
    r = run_panel_did(bench, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['category'] = '创新数量'
    quantity_results.append(r)
    if 'error' not in r:
        coef = r['manufacturing_post2021_coef']
        se = r['manufacturing_post2021_se']
        p = r['manufacturing_post2021_p']
        print(f'{dep:30s}: coef={coef:+8.4f}, se={se:.4f}, p={p:.4f} {fmt_sig(p)}')

quantity_df = pd.DataFrame(quantity_results)
quantity_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_quantity_results.csv'), index=False)
print(f'\n结果已保存: outputs/focus_quantity_results.csv')

## 5. 研发投入调整模型

In [ ]:
rd_deps = ['ln_rd_expense_10k', 'rd_intensity_01', 'ln_rd_staff', 'rd_staff_ratio_01']
rd_results = []

for dep in rd_deps:
    r = run_panel_did(bench, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['category'] = '研发投入'
    rd_results.append(r)
    if 'error' not in r:
        coef = r['manufacturing_post2021_coef']
        se = r['manufacturing_post2021_se']
        p = r['manufacturing_post2021_p']
        print(f'{dep:30s}: coef={coef:+8.4f}, se={se:.4f}, p={p:.4f} {fmt_sig(p)}')

rd_df = pd.DataFrame(rd_results)
rd_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_rd_adjustment_results.csv'), index=False)

# 制造业研发趋势 (绝对水平)
print('\n制造业研发投入绝对趋势 (均值):')
mfg_trend = bench[bench['manufacturing'] == 1].groupby('year').agg(
    rd_expense_mean=('rd_expense', lambda x: x.mean() / 1e8),  # 亿元
    rd_intensity_mean=('rd_intensity', 'mean'),                 # %
    rd_staff_mean=('rd_staff', 'mean'),
    invention_apply_mean=('invention_apply', 'mean'),
    n=('stock_code', 'count'),
).reset_index()
display(mfg_trend)

## 6. 创新效率主模型（核心展示）

In [ ]:
eff_deps = ['eff_apply_rd_10k', 'eff_grant_rd_10k', 'eff_apply_staff', 'eff_grant_staff']
eff_results = []

for dep in eff_deps:
    r = run_panel_did(bench, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['category'] = '创新效率'
    eff_results.append(r)
    if 'error' not in r:
        coef = r['manufacturing_post2021_coef']
        se = r['manufacturing_post2021_se']
        p = r['manufacturing_post2021_p']
        print(f'{dep:30s}: coef={coef:+8.4f}, se={se:.4f}, p={p:.4f} {fmt_sig(p)}')

eff_df = pd.DataFrame(eff_results)
eff_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_efficiency_main_results.csv'), index=False)

## 7. 效率拆解综合表

核心恒等式: DID(eff_apply_rd_10k) ≡ DID(ln_invention_apply) − DID(ln_rd_expense_10k)

In [ ]:
decomp_vars = [
    ('ln_invention_apply', '创新数量'),
    ('ln_invention_grant', '创新数量'),
    ('ln_rd_expense_10k', '研发投入'),
    ('rd_intensity_01', '研发投入'),
    ('ln_rd_staff', '研发投入'),
    ('eff_apply_rd_10k', '创新效率'),
    ('eff_grant_rd_10k', '创新效率'),
    ('eff_apply_staff', '创新效率'),
    ('eff_grant_staff', '创新效率'),
]

decomp_all = []
for dep, cat in decomp_vars:
    r = run_panel_did(bench, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['category'] = cat
    decomp_all.append(r)
    if 'error' not in r:
        coef = r['manufacturing_post2021_coef']
        se = r['manufacturing_post2021_se']
        p = r['manufacturing_post2021_p']
        print(f'{dep:30s} [{cat}]: coef={coef:+8.4f}, se={se:.4f}, p={p:.4f} {fmt_sig(p)}')

decomp_df = pd.DataFrame(decomp_all)
decomp_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_efficiency_decomposition.csv'), index=False)

In [ ]:
# 恒等式验证
did_inv = decomp_df[decomp_df['dependent'] == 'ln_invention_apply']['manufacturing_post2021_coef'].values[0]
did_rd = decomp_df[decomp_df['dependent'] == 'ln_rd_expense_10k']['manufacturing_post2021_coef'].values[0]
did_eff = decomp_df[decomp_df['dependent'] == 'eff_apply_rd_10k']['manufacturing_post2021_coef'].values[0]

print('DID 恒等式验证:')
print(f'  DID(ln_invention_apply)        = {did_inv:+.4f}')
print(f'  DID(ln_rd_expense_10k)         = {did_rd:+.4f}')
print(f'  DID(eff_apply_rd_10k)          = {did_eff:+.4f}')
print(f'  DID(inv) − DID(rd)             = {did_inv - did_rd:+.4f}')
print(f'  ≈ DID(eff) ?                   = {"完全一致 ✓" if abs((did_inv - did_rd) - did_eff) < 1e-8 else "存在差异"}')
print(f'\n结论: 效率提升 {did_eff:+.4f} ≈ 创新产出 {did_inv:+.4f} − 研发投入 ({did_rd:+.4f})')

## 8. 异质性一：高研发基础企业

In [ ]:
het_high_rd_results = []
bench_het = bench.dropna(subset=['high_pre_rd']).copy()
bench_het['did_x_high_pre_rd'] = bench_het['manufacturing_post2021'] * bench_het['high_pre_rd']

# 交互项模型
print('=== 交互项模型 ===')
for dep in eff_deps:
    r = run_panel_did(bench_het, dep,
                      ['manufacturing_post2021', 'did_x_high_pre_rd'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['model'] = '交互项'
    het_high_rd_results.append(r)
    if 'error' not in r:
        for v in ['manufacturing_post2021', 'did_x_high_pre_rd']:
            coef = r.get(f'{v}_coef', np.nan)
            p = r.get(f'{v}_p', np.nan)
            print(f'  {dep:25s} {v:30s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

# 分组回归
print('\n=== 分组回归 ===')
for group_val, group_label in [(1, '高研发基础'), (0, '低研发基础')]:
    sub = bench_het[bench_het['high_pre_rd'] == group_val].copy()
    for dep in eff_deps:
        r = run_panel_did(sub, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
        r['dependent'] = dep
        r['model'] = f'分组-{group_label}'
        het_high_rd_results.append(r)
        if 'error' not in r:
            coef = r['manufacturing_post2021_coef']
            p = r['manufacturing_post2021_p']
            print(f'  {group_label:10s} {dep:25s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

het_high_rd_df = pd.DataFrame(het_high_rd_results)
het_high_rd_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_high_rd_heterogeneity.csv'), index=False)

## 9. 异质性二：制造业内部研发暴露

In [ ]:
mfg_only = bench_het[bench_het['manufacturing'] == 1].copy()
mfg_only['high_pre_rd_post2021'] = mfg_only['high_pre_rd'] * mfg_only['post2021']
mfg_only['high_pre_rd_q4_post2021'] = mfg_only['high_pre_rd_q4'] * mfg_only['post2021']

within_mfg_results = []

for dep in eff_deps:
    for treat_var in ['high_pre_rd_post2021', 'high_pre_rd_q4_post2021']:
        r = run_panel_did(mfg_only, dep, [treat_var] + BASE_CONTROLS)
        r['dependent'] = dep
        r['treatment'] = treat_var
        within_mfg_results.append(r)
        if 'error' not in r:
            coef = r.get(f'{treat_var}_coef', np.nan)
            p = r.get(f'{treat_var}_p', np.nan)
            print(f'{dep:25s} {treat_var:35s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

within_mfg_df = pd.DataFrame(within_mfg_results)
within_mfg_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_within_manufacturing_exposure.csv'), index=False)

## 10. 异质性三：所有制

In [ ]:
ownership_results = []
bench_soe = bench.dropna(subset=['soe']).copy()
bench_soe['did_x_private'] = bench_soe['manufacturing_post2021'] * bench_soe['private']

# 交互项模型
print('=== 交互项模型 (非国企交互) ===')
for dep in eff_deps:
    r = run_panel_did(bench_soe, dep,
                      ['manufacturing_post2021', 'did_x_private'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['model'] = '交互项-非国企'
    ownership_results.append(r)
    if 'error' not in r:
        for v in ['manufacturing_post2021', 'did_x_private']:
            coef = r.get(f'{v}_coef', np.nan)
            p = r.get(f'{v}_p', np.nan)
            print(f'  {dep:25s} {v:30s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

# 分组回归
print('\n=== 分组回归 ===')
for group_val, group_label in [(0, '国企'), (1, '非国企')]:
    sub = bench_soe[bench_soe['soe'] == group_val].copy()
    for dep in eff_deps:
        r = run_panel_did(sub, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
        r['dependent'] = dep
        r['model'] = f'分组-{group_label}'
        ownership_results.append(r)
        if 'error' not in r:
            coef = r['manufacturing_post2021_coef']
            p = r['manufacturing_post2021_p']
            print(f'  {group_label:10s} {dep:25s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

ownership_df = pd.DataFrame(ownership_results)
ownership_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_ownership_heterogeneity.csv'), index=False)

## 11. 政策暴露强度模型

In [ ]:
exposure_results = []

for dep in eff_deps:
    r = run_panel_did(bench, dep,
                      ['policy_exposure', 'manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    exposure_results.append(r)
    if 'error' not in r:
        for v in ['policy_exposure', 'manufacturing_post2021']:
            coef = r.get(f'{v}_coef', np.nan)
            p = r.get(f'{v}_p', np.nan)
            print(f'{dep:25s} {v:30s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

exposure_df = pd.DataFrame(exposure_results)
exposure_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_policy_exposure_results.csv'), index=False)

## 12. 稳健性检验

In [ ]:
robustness_summary = []

# ---------- 12.1 效率指标口径稳健性 ----------
print('--- 12.1 效率指标口径稳健性 ---')
eff_units = [
    ('eff_apply_rd_10k', '万元对数'),
    ('eff_grant_rd_10k', '万元对数'),
    ('eff_apply_staff', '人员对数'),
    ('eff_grant_staff', '人员对数'),
]
for dep, label in eff_units:
    r = run_panel_did(bench, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['robustness_type'] = 'unit'
    r['label'] = label
    robustness_summary.append(r)

# ---------- 12.2 替代效率指标 (asinh比率型) ----------
print('\n--- 12.2 替代效率指标 (asinh比率) ---')
asinh_deps = ['asinh_apply_per_rd', 'asinh_grant_per_rd',
              'asinh_apply_per_staff', 'asinh_grant_per_staff']
for dep in asinh_deps:
    r = run_panel_did(bench, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r['dependent'] = dep
    r['robustness_type'] = 'asinh_ratio'
    robustness_summary.append(r)
    if 'error' not in r:
        coef = r['manufacturing_post2021_coef']
        p = r['manufacturing_post2021_p']
        print(f'  {dep:30s}: coef={coef:+8.4f}, p={p:.4f} {fmt_sig(p)}')

In [ ]:
# ---------- 12.3 强固定效应 ----------
print('--- 12.3 强固定效应 ---')
bench_fe = bench.dropna(subset=['province_clean', 'industry_code']).copy()
bench_fe['province_year'] = bench_fe['province_clean'].astype(str) + '_' + bench_fe['year'].astype(str)
bench_fe['industry_year'] = bench_fe['industry_code'].astype(str) + '_' + bench_fe['year'].astype(str)

stronger_deps = eff_deps + ['ln_rd_expense_10k', 'ln_rd_staff']

for dep in stronger_deps:
    # Firm+Year FE (baseline)
    r_base = run_panel_did(bench_fe, dep, ['manufacturing_post2021'] + BASE_CONTROLS)
    r_base['dependent'] = dep
    r_base['fe_spec'] = 'Firm+Year'
    r_base['robustness_type'] = 'stronger_fe'
    robustness_summary.append(r_base)

    # Firm+Year+Prov×Year
    sub = bench_fe.dropna(subset=[dep, 'manufacturing_post2021'] + BASE_CONTROLS).copy()
    for fe_label, extra_col in [('Prov×Year', 'province_year'), ('Ind×Year', 'industry_year')]:
        try:
            dummies = pd.get_dummies(sub[extra_col], drop_first=True)
            dummies.index = sub.index

            if extra_col == 'industry_year':
                use_time_effects = False
            else:
                use_time_effects = True

            sub_idx = sub.set_index(['stock_code', 'year'])
            dummies_idx = dummies.copy()
            dummies_idx.index = sub_idx.index

            exog = sub_idx[['manufacturing_post2021'] + BASE_CONTROLS].join(dummies_idx)
            mod = PanelOLS(
                dependent=sub_idx[dep],
                exog=exog,
                entity_effects=True,
                time_effects=use_time_effects,
                drop_absorbed=True,
            )
            res = mod.fit(cov_type='clustered', cluster_entity=True)
            coef_val = res.params.get('manufacturing_post2021', np.nan)
            se_val = res.std_errors.get('manufacturing_post2021', np.nan)
            p_val = res.pvalues.get('manufacturing_post2021', np.nan)
            robustness_summary.append({
                'dependent': dep,
                'fe_spec': f'Firm+{"Year+" if use_time_effects else ""}{fe_label}',
                'robustness_type': 'stronger_fe',
                'manufacturing_post2021_coef': coef_val,
                'manufacturing_post2021_se': se_val,
                'manufacturing_post2021_p': p_val,
                'N': len(sub_idx),
                'firms': sub_idx.index.get_level_values(0).nunique(),
            })
            print(f'  {dep:25s} x Firm+{"Year+" if use_time_effects else ""}{fe_label:25s}: '
                  f'coef={coef_val:+8.4f}, p={p_val:.4f} {fmt_sig(p_val)}')
        except Exception as e:
            print(f'  {dep:25s} x Firm+Year+{fe_label:25s}: ERROR - {str(e)[:80]}')

In [ ]:
# ---------- 12.4 安慰剂检验 ----------
print('--- 12.4 安慰剂检验 (2017-2020) ---')
placebo_sample = bench[bench['year'] <= 2020].copy()
placebo_sample['placebo_did2019'] = placebo_sample['manufacturing'] * (placebo_sample['year'] >= 2019).astype(int)
placebo_sample['placebo_did2020'] = placebo_sample['manufacturing'] * (placebo_sample['year'] >= 2020).astype(int)

placebo_deps = eff_deps + ['ln_invention_apply', 'ln_rd_expense_10k']
placebo_results = []

for dep in placebo_deps:
    for pvar in ['placebo_did2019', 'placebo_did2020']:
        r = run_panel_did(placebo_sample, dep, [pvar] + BASE_CONTROLS)
        r['dependent'] = dep
        r['placebo_var'] = pvar
        placebo_results.append(r)
        if 'error' not in r:
            coef = r.get(f'{pvar}_coef', np.nan)
            p = r.get(f'{pvar}_p', np.nan)
            flag = '⚠️ 显著!' if p < 0.1 else '✓ 不显著'
            print(f'  {dep:25s} x {pvar:20s}: coef={coef:+8.4f}, p={p:.4f} {flag}')

placebo_df = pd.DataFrame(placebo_results)
placebo_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_placebo.csv'), index=False)

In [ ]:
# ---------- 12.5 事件研究 ----------
print('--- 12.5 事件研究 (基准年=2020) ---')
event_years = [2017, 2018, 2019, 2021, 2022]
for y in event_years:
    bench[f'event_{y}'] = bench['manufacturing'] * (bench['year'] == y).astype(int)

event_vars = [f'event_{y}' for y in event_years]
event_deps = ['ln_invention_apply', 'ln_rd_expense_10k', 'ln_rd_staff',
              'eff_apply_rd_10k', 'eff_apply_staff']

event_all = []
for dep in event_deps:
    r = run_panel_did(bench, dep, event_vars + BASE_CONTROLS)
    if 'error' not in r:
        print(f'\n{dep}: N={r["N"]}')
        for ev in event_vars:
            coef = r.get(f'{ev}_coef', np.nan)
            se = r.get(f'{ev}_se', np.nan)
            p = r.get(f'{ev}_p', np.nan)
            event_all.append({
                'dependent': dep,
                'event_year': ev,
                'coef': coef,
                'se': se,
                'p': p,
                'ci_lower': coef - 1.96 * se if not pd.isna(coef) else np.nan,
                'ci_upper': coef + 1.96 * se if not pd.isna(coef) else np.nan,
                'N': r.get('N', 0),
            })
            print(f'  {ev}: coef={coef:+7.4f}, se={se:.4f}, p={p:.4f} {fmt_sig(p)}')

event_df = pd.DataFrame(event_all)
event_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_event_study.csv'), index=False)

# 保存稳健性汇总
robustness_summary_df = pd.DataFrame(robustness_summary)
robustness_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'focus_robustness_summary.csv'), index=False)
print('\n稳健性检验结果已保存。')

## 13. 图表生成

以下生成12张学术彩色图表，配色采用低饱和度学术风格。输出到 `outputs/figures/`。


In [ ]:
# ---------- 趋势数据准备 ----------
trend_data = v4[(v4['year'] >= 2017) & (v4['year'] <= 2024)].copy()
trend_data['ln_rd_expense_10k'] = np.log1p(trend_data['rd_expense'] / 10000)
trend_data['eff_apply_rd_10k'] = trend_data['ln_invention_apply'] - trend_data['ln_rd_expense_10k']
trend_data['eff_apply_staff'] = trend_data['ln_invention_apply'] - trend_data['ln_rd_staff']

def add_policy_vline(ax, x=2021):
    """在x位置添加政策冲击竖线"""
    ax.axvline(x=x, color=C_POLICY, linestyle='--', linewidth=1.5, alpha=0.8)

def add_zero_hline(ax):
    """添加y=0水平线"""
    ax.axhline(y=0, color='black', linewidth=0.8)

# 制造业年度趋势汇总
mfg = bench[bench['manufacturing'] == 1]
mfg_a = mfg.groupby('year').agg(
    rd=('rd_expense', lambda x: x.mean() / 1e8),
    rd_int=('rd_intensity', 'mean'),
    inv_app=('invention_apply', 'mean'),
    inv_grt=('invention_grant', 'mean'),
).reset_index()

print('数据准备完成。')


In [ ]:
# ============================================================
# 图1: 样本结构图（饼图 + 分年堆叠柱状图）
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# 饼图
sizes = [16740, 10032]
labels = ['制造业\n(16,740条, 62.5%)', '非制造业\n(10,032条, 37.5%)']
colors_pie = [C_MFG, C_NONMFG]
wedges, texts, autotexts = ax1.pie(sizes, labels=None, colors=colors_pie, startangle=90,
                                     autopct='%1.1f%%', pctdistance=0.6,
                                     wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
                                     textprops={'fontsize': 9})
for autotext in autotexts:
    autotext.set_fontweight('bold')
ax1.legend(wedges, labels, loc='lower center', fontsize=9, frameon=True, fancybox=True)
ax1.set_title('样本结构', fontsize=13, fontweight='bold', color=C_DARK)

# 分年柱状图
yearly = bench.groupby('year')['manufacturing'].agg(['sum', 'count']).reset_index()
yearly['nonmfg'] = yearly['count'] - yearly['sum']
yearly['mfg_pct'] = yearly['sum'] / yearly['count'] * 100
x = yearly['year'].values
ax2.bar(x, yearly['nonmfg'], color=C_NONMFG, label='非制造业', edgecolor='white', linewidth=0.5)
ax2.bar(x, yearly['sum'], bottom=yearly['nonmfg'], color=C_MFG, label='制造业', edgecolor='white', linewidth=0.5)
for i, yr in enumerate(x):
    ax2.text(yr, yearly['count'].values[i] + 100, f"{yearly['mfg_pct'].values[i]:.1f}%",
             ha='center', fontsize=8, fontweight='bold', color=C_DARK)
ax2.set_title('分年度样本构成', fontsize=13, fontweight='bold', color=C_DARK)
ax2.set_xlabel('年份'); ax2.set_ylabel('观测数')
ax2.legend(fontsize=9, frameon=True, fancybox=True)
ax2.set_xticks(x); ax2.set_ylim(0, yearly['count'].max()*1.18)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
fig.suptitle('图 1  样本结构图', fontsize=14, fontweight='bold', color=C_DARK, y=1.01)
plt.tight_layout()
save_fig(fig, 'fig01_sample_structure')

# ============================================================
# 图2: 制造业研发投入与专利产出年度趋势图（4面板）
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
panels = [
    (axes[0,0], 'rd', '研发支出（亿元）', C_BLUE),
    (axes[0,1], 'rd_int', '研发强度（%）', C_GREEN),
    (axes[1,0], 'inv_app', '发明专利申请（件）', C_ORANGE),
    (axes[1,1], 'inv_grt', '发明专利授权（件）', C_TEAL),
]
for ax, col, ylabel, color in panels:
    ax.plot(mfg_a['year'], mfg_a[col], color=color, linewidth=2.8,
            marker='D', markersize=9, markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=color)
    add_policy_vline(ax)
    ax.text(2021.08, ax.get_ylim()[1]*0.92, '2021年\n政策时点', color=C_POLICY, fontsize=8, va='top', fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=10, color=C_DARK); ax.set_xlabel('年份', fontsize=10)
    ax.set_xticks([2017,2018,2019,2020,2021,2022])
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    for _, row in mfg_a.iterrows():
        ax.annotate(f'{row[col]:.1f}', (int(row['year']), row[col]), textcoords='offset points',
                    xytext=(0,12), ha='center', fontsize=7.5, color=C_DARK,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
fig.suptitle('图 2  制造业研发投入与专利产出年度趋势图', fontsize=14, fontweight='bold', color=C_DARK, y=1.01)
plt.tight_layout()
save_fig(fig, 'fig02_mfg_trend')

# ============================================================
# 图3: 制造业与非制造业主要变量趋势对比图（3面板）
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
trend_cfg = [
    ('ln_rd_expense_10k', 'ln(研发支出)', 'A. 研发支出（万元对数）'),
    ('ln_invention_apply', 'ln(发明申请)', 'B. 发明专利申请（对数）'),
    ('eff_apply_rd_10k', '发明申请/研发支出效率', 'C. 创新效率（对数差分）'),
]
for ax, (var, ylabel, title) in zip(axes, trend_cfg):
    for mfg_val, label, color, marker, ls in [(1,'制造业',C_BLUE,'D','-'), (0,'非制造业',C_ORANGE,'o','--')]:
        yearly = bench[bench['manufacturing']==mfg_val].groupby('year')[var].mean()
        ax.plot(yearly.index, yearly.values, color=color, linestyle=ls, linewidth=2.5,
                marker=marker, markersize=8, markerfacecolor='white', markeredgewidth=2.5, label=label)
    add_policy_vline(ax)
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_DARK)
    ax.set_xlabel('年份'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9, frameon=True, fancybox=True, loc='upper left')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xticks([2017,2018,2019,2020,2021,2022])
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle('图 3  制造业与非制造业主要变量趋势对比图', fontsize=14, fontweight='bold', color=C_DARK, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig03_mfg_vs_nonmfg')


In [ ]:
# ============================================================
# 图4: 主要变量分布图（3面板直方图）
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
dist_cfg = [
    ('ln_invention_apply', 'ln(发明专利申请)', 'A. 发明专利申请', C_BLUE),
    ('ln_rd_expense_10k', 'ln(研发支出, 万元)', 'B. 研发支出', C_ORANGE),
    ('eff_apply_rd_10k', '发明申请/研发支出效率', 'C. 创新效率', C_GREEN),
]
for ax, (var, xlabel, title, color) in zip(axes, dist_cfg):
    s = bench[var].dropna()
    lo, hi = s.quantile(0.01), s.quantile(0.99)
    display = s[(s >= lo) & (s <= hi)]
    ax.hist(display, bins=60, color=color, alpha=0.6, edgecolor='white', linewidth=0.5)
    ax.axvline(display.median(), color=C_RED, linestyle='--', linewidth=2, label=f'中位数: {display.median():.2f}')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_DARK)
    ax.set_xlabel(xlabel); ax.set_ylabel('频数')
    ax.legend(fontsize=9, frameon=True, fancybox=True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle('图 4  主要变量分布图', fontsize=14, fontweight='bold', color=C_DARK, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig04_distributions')

# ============================================================
# 图5: 政策前后分组均值变化图（3面板分组柱状图）
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
pp_cfg = [
    ('ln_rd_expense_10k', 'ln(研发支出)', 'A. 研发支出'),
    ('ln_invention_apply', 'ln(发明申请)', 'B. 发明专利申请'),
    ('eff_apply_rd_10k', '发明申请/研发支出效率', 'C. 创新效率'),
]
width = 0.35
for ax, (var, ylabel, title) in zip(axes, pp_cfg):
    groups = {}
    for mfg_val, mfg_label, color in [(1,'制造业',C_BLUE),(0,'非制造业',C_ORANGE)]:
        for post_val, post_label, hatch in [(0,'政策前',''),(1,'政策后','//')]:
            subset = bench[(bench['manufacturing']==mfg_val)&(bench['post2021']==post_val)]
            groups[f'{mfg_label}-{post_label}'] = (subset[var].mean(), color, hatch)
    x = np.arange(2)
    ax.bar(x-width/2, [groups['制造业-政策前'][0],groups['非制造业-政策前'][0]], width,
           color=[C_BLUE,C_ORANGE], edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.bar(x+width/2, [groups['制造业-政策后'][0],groups['非制造业-政策后'][0]], width,
           color=[C_BLUE,C_ORANGE], edgecolor='white', linewidth=0.8, alpha=0.85, hatch='//')
    ax.set_xticks(x); ax.set_xticklabels(['制造业','非制造业'], fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_DARK); ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
legend_el = [Patch(facecolor=C_BLUE, alpha=0.85, label='制造业 政策前'),
             Patch(facecolor=C_ORANGE, alpha=0.85, label='非制造业 政策前'),
             Patch(facecolor=C_BLUE, alpha=0.85, hatch='//', label='制造业 政策后'),
             Patch(facecolor=C_ORANGE, alpha=0.85, hatch='//', label='非制造业 政策后')]
fig.legend(handles=legend_el, loc='lower center', ncol=4, fontsize=9, frameon=True, fancybox=True, bbox_to_anchor=(0.5,-0.04))
fig.suptitle('图 5  政策前后分组均值变化图', fontsize=14, fontweight='bold', color=C_DARK, y=1.04)
plt.tight_layout()
save_fig(fig, 'fig05_prepost_means')


In [ ]:
# ============================================================
# 图6: 基准DID估计结果森林图
# ============================================================
fig, ax = plt.subplots(figsize=(10.5, 7.5))
forest_data = [
    ('发明专利申请', -0.0247, 0.0246, 0.3146, 'A. 创新数量'),
    ('发明专利授权', -0.0113, 0.0230, 0.6224, 'A. 创新数量'),
    ('专利总申请', -0.0017, 0.0339, 0.9591, 'A. 创新数量'),
    ('专利总授权', 0.0348, 0.0357, 0.3298, 'A. 创新数量'),
    ('研发支出（万元对数）', -0.2884, 0.0568, 0.0000, 'B. 研发投入'),
    ('研发强度 (0-1)', -0.0029, 0.0013, 0.0245, 'B. 研发投入'),
    ('研发人员（对数）', -0.1971, 0.0443, 0.0000, 'B. 研发投入'),
    ('研发人员占比 (0-1)', 0.0002, 0.0029, 0.9509, 'B. 研发投入'),
    ('发明申请 / 研发支出效率', 0.2637, 0.0616, 0.0000, 'C. 创新效率'),
    ('发明授权 / 研发支出效率', 0.2771, 0.0611, 0.0000, 'C. 创新效率'),
    ('发明申请 / 研发人员效率', 0.1724, 0.0499, 0.0005, 'C. 创新效率'),
    ('发明授权 / 研发人员效率', 0.1858, 0.0491, 0.0002, 'C. 创新效率'),
]
n = len(forest_data)
# 分类背景
cat_colors = {'A. 创新数量': '#E8E8E8', 'B. 研发投入': '#F0E8D8', 'C. 创新效率': '#E0F0E0'}
for cat, bg in cat_colors.items():
    indices = [i for i, d in enumerate(forest_data) if d[4] == cat]
    if indices:
        lo, hi = min(indices), max(indices)
        ax.axhspan(n-1-hi-0.5, n-1-lo+0.5, alpha=0.25, facecolor=bg, zorder=0, edgecolor='#CCCCCC', linewidth=0.5)
        ax.text(0.98, n-1-(lo+hi)/2, cat, transform=ax.get_yaxis_transform(), ha='right', fontsize=9, fontweight='bold', va='center', color=C_DARK)

for i, (label, coef, se, p, cat) in enumerate(forest_data):
    ci_low, ci_high = coef - 1.96*se, coef + 1.96*se
    if p < 0.01:
        color, marker, ms = (C_SIG_POS if coef>0 else C_SIG_NEG), 'D', 8
    elif p < 0.05:
        color, marker, ms = (C_SIG_POS if coef>0 else C_SIG_NEG), 's', 7
    elif p < 0.10:
        color, marker, ms = '#A0A0A0', ('^' if coef>0 else 'v'), 6
    else:
        color, marker, ms = C_GRAY, 'o', 6
    ax.errorbar(coef, n-1-i, xerr=[[coef-ci_low],[ci_high-coef]], fmt=marker, color=color,
                capsize=3, markersize=ms, markeredgecolor='white', markeredgewidth=0.5, linewidth=2)
    sig_str = fmt_sig(p)
    x_txt = coef + (0.03 if coef>=0 else -0.03)
    ha = 'left' if coef>=0 else 'right'
    ax.text(x_txt, n-1-i, f'{coef:+.3f}{sig_str}', va='center', ha=ha, fontsize=8.5,
            fontweight='bold' if p<0.05 else 'normal', color=C_DARK)
ax.set_yticks(range(n))
ax.set_yticklabels([d[0] for d in forest_data], fontsize=8.5)
add_zero_hline(ax)
ax.set_xlabel('DID 系数（制造业 × 政策后）', fontsize=11, color=C_DARK)
ax.set_title('图 6  基准 DID 估计结果森林图', fontsize=14, fontweight='bold', color=C_DARK)
ax.grid(True, alpha=0.25, axis='x', linestyle='--')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
legend_el = [Line2D([0],[0],marker='D',color='w',markerfacecolor=C_SIG_POS,markersize=9,label='p<0.01'),
             Line2D([0],[0],marker='s',color='w',markerfacecolor=C_SIG_POS,markersize=7,label='p<0.05'),
             Line2D([0],[0],marker='o',color='w',markerfacecolor=C_GRAY,markersize=6,label='不显著')]
ax.legend(handles=legend_el, fontsize=8, loc='lower right', frameon=True, fancybox=True)
plt.tight_layout()
save_fig(fig, 'fig06_did_forest')


In [ ]:
# ============================================================
# 图7: 创新效率来源拆解图（瀑布图风格）
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5.5))
vals = [-0.0247, 0.2884, 0.2637]
labels = ['创新产出\nDID(ln发明申请)\n−0.025', '研发投入调整\n−DID(ln研发支出)\n+0.288', '创新效率\nDID(发明申请/研发支出)\n+0.264']
colors_bar = [C_GRAY, C_ORANGE, C_GREEN]
ax.bar(0, vals[0], color=colors_bar[0], edgecolor='white', linewidth=1, width=0.5, alpha=0.85)
ax.bar(1, vals[1], bottom=vals[0], color=colors_bar[1], edgecolor='white', linewidth=1, width=0.5, alpha=0.85, hatch='//')
ax.bar(2, vals[2], color=colors_bar[2], edgecolor='white', linewidth=1, width=0.5, alpha=0.85)
# 连接线
ax.plot([0.25,0.75], [vals[0],vals[0]], '-', color=C_DARK, linewidth=1.2)
ax.plot([0.25,0.75], [vals[0],vals[0]+vals[1]], '--', color=C_DARK, linewidth=1, alpha=0.6)
# 标注
for i, (val, label) in enumerate(zip(vals, labels)):
    mid = (vals[0]+val/2) if i==1 else (val/2 if val<0 else val*0.4)
    ax.text(i, mid, f'{val:+.4f}', ha='center', va='center', fontsize=13, fontweight='bold', color='black')
ax.set_xticks([0,1,2]); ax.set_xticklabels(labels, fontsize=9, color=C_DARK)
add_zero_hline(ax)
ax.set_ylabel('DID 系数', fontsize=11, color=C_DARK)
ax.set_title('图 7  创新效率来源拆解图', fontsize=14, fontweight='bold', color=C_DARK)
ax.grid(True, alpha=0.25, axis='y', linestyle='--')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.text(0.5, -0.18, '效率指标改善 ≈ 研发投入相对增长较慢 − 专利数量相对变化不显著', transform=ax.transAxes,
        ha='center', fontsize=9, fontstyle='italic', color=C_DARK,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E1', alpha=0.8, edgecolor=C_GOLD))
plt.tight_layout()
save_fig(fig, 'fig07_efficiency_decomp')


In [ ]:
# ============================================================
# 图8: 异质性分析结果图（热力图）
# ============================================================
fig, ax = plt.subplots(figsize=(10.5, 4.8))
het_data = [
    ('高研发基础\n交互项', '发明申请/\n研发支出', 0.2253, 0.0509, 0.0000),
    ('高研发基础\n交互项', '发明授权/\n研发支出', 0.2718, 0.0484, 0.0000),
    ('高研发基础\n交互项', '发明申请/\n研发人员', 0.1368, 0.0534, 0.0104),
    ('高研发基础\n交互项', '发明授权/\n研发人员', 0.1833, 0.0512, 0.0003),
    ('制造业内部\n高研发暴露', '发明申请/\n研发支出', 0.1977, 0.0503, 0.0001),
    ('制造业内部\n高研发暴露', '发明授权/\n研发支出', 0.2445, 0.0476, 0.0000),
    ('制造业内部\n高研发暴露', '发明申请/\n研发人员', 0.1236, 0.0537, 0.0215),
    ('制造业内部\n高研发暴露', '发明授权/\n研发人员', 0.1704, 0.0514, 0.0009),
]
rows = sorted(set(d[0] for d in het_data), reverse=True)
cols = sorted(set(d[1] for d in het_data))
n_rows, n_cols = len(rows), len(cols)
matrix = np.zeros((n_rows, n_cols))
annot = [['' for _ in range(n_cols)] for _ in range(n_rows)]
for row_label, col_label, coef, se, p in het_data:
    ri, ci = rows.index(row_label), cols.index(col_label)
    matrix[ri, ci] = coef
    annot[ri][ci] = f'{coef:.3f}\n{fmt_sig(p)}'
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0.08, vmax=0.30)
for i in range(n_rows):
    for j in range(n_cols):
        txt_color = 'white' if matrix[i,j] > 0.22 else 'black'
        ax.text(j, i, annot[i][j], ha='center', va='center', fontsize=11, fontweight='bold', color=txt_color)
ax.set_xticks(range(n_cols)); ax.set_xticklabels(cols, fontsize=10)
ax.set_yticks(range(n_rows)); ax.set_yticklabels(rows, fontsize=10)
cbar = plt.colorbar(im, ax=ax, shrink=0.9, pad=0.02)
cbar.set_label('DID 系数', fontsize=10, color=C_DARK)
ax.set_title('图 8  异质性分析结果图', fontsize=14, fontweight='bold', color=C_DARK)
plt.tight_layout()
save_fig(fig, 'fig08_heterogeneity')


In [ ]:
# ============================================================
# 图9: 基准模型与强固定效应模型结果对比图
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5.5))
fe_data = [
    ('发明申请/\n研发支出', 0.2637, 0.0616, 0.2552, 0.0635),
    ('发明授权/\n研发支出', 0.2771, 0.0611, 0.2903, 0.0632),
    ('发明申请/\n研发人员', 0.1724, 0.0499, 0.1614, 0.0512),
    ('发明授权/\n研发人员', 0.1858, 0.0491, 0.1965, 0.0506),
]
x = np.arange(len(fe_data))
width = 0.32
for i, (label, bc, bs, sc, ss) in enumerate(fe_data):
    ax.bar(i-width/2, bc, width, yerr=bs*1.96, color=C_BLUE, edgecolor='white', linewidth=0.8,
           capsize=5, alpha=0.85, label='基准模型' if i==0 else '')
    ax.bar(i+width/2, sc, width, yerr=ss*1.96, color=C_PURPLE, edgecolor='white', linewidth=0.8,
           capsize=5, alpha=0.85, hatch='//', label='+Prov×Year' if i==0 else '')
ax.set_xticks(x); ax.set_xticklabels([d[0] for d in fe_data], fontsize=10, color=C_DARK)
add_zero_hline(ax)
ax.set_ylabel('DID 系数', fontsize=11, color=C_DARK)
ax.set_title('图 9  基准模型与强固定效应模型结果对比图', fontsize=14, fontweight='bold', color=C_DARK)
ax.legend(fontsize=9, frameon=True, fancybox=True, loc='upper right')
ax.grid(True, alpha=0.25, axis='y', linestyle='--')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, 'fig09_stronger_fe')


In [ ]:
# ============================================================
# 图10: 事件研究结果图（3面板）
# ============================================================
ed = {
    'A. 发明专利申请': {
        2017:(0.0031,0.0333), 2018:(-0.0319,0.0311), 2019:(-0.0142,0.0302),
        2020:(0.0,0.0), 2021:(-0.0334,0.0278), 2022:(-0.0352,0.0314),
        'color':C_BLUE, 'ylim':(-0.18,0.12)},
    'B. 研发支出': {
        2017:(0.6871,0.1034), 2018:(0.3962,0.0774), 2019:(0.2963,0.0575),
        2020:(0.0,0.0), 2021:(0.0101,0.0550), 2022:(-0.0021,0.0661),
        'color':C_ORANGE, 'ylim':(-0.25,1.0)},
    'C. 创新效率': {
        2017:(-0.6840,0.1074), 2018:(-0.4281,0.0829), 2019:(-0.3104,0.0642),
        2020:(0.0,0.0), 2021:(-0.0435,0.0610), 2022:(-0.0330,0.0724),
        'color':C_GREEN, 'ylim':(-1.0,0.25)},
}
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, (title, data) in zip(axes, ed.items()):
    year_keys = [k for k in data.keys() if isinstance(k, int)]
    py = [y for y in sorted(year_keys) if y != 2020]
    pc = [data[y][0] for y in py]
    ps = [data[y][1] for y in py]
    ci_l = [c-1.96*s for c,s in zip(pc,ps)]
    ci_h = [c+1.96*s for c,s in zip(pc,ps)]
    color = data['color']
    ax.errorbar(py, pc, yerr=[np.array(pc)-np.array(ci_l), np.array(ci_h)-np.array(pc)],
                fmt='D-', color=color, capsize=5, markersize=9,
                markerfacecolor='white', markeredgewidth=2.5, linewidth=2.5, label='事件系数')
    ax.plot(2020, 0, 'D', color=C_RED, markersize=12, markerfacecolor=C_RED, label='2020（基准年）')
    add_zero_hline(ax)
    ax.axvline(x=2020.5, color=C_RED, linestyle='--', linewidth=2, alpha=0.7)
    ax.axvspan(2016.5, 2020.5, alpha=0.06, facecolor=C_GRAY)
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_DARK)
    ax.set_xlabel('年份'); ax.set_xticks([2017,2018,2019,2020,2021,2022])
    ax.set_ylim(data['ylim'])
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=8, frameon=True, fancybox=True, loc='upper left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle('图 10  事件研究结果图（基准年 = 2020）', fontsize=14, fontweight='bold', color=C_DARK, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig10_event_study')


In [ ]:
# ============================================================
# 图11: 安慰剂检验结果图
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
pl_data = [
    ('真实政策时点\n2021年', 0.2637, 0.0616, C_GREEN),
    ('假想政策时点\n2019年', 0.2902, 0.0719, C_ORANGE),
    ('假想政策时点\n2020年', 0.3252, 0.0701, C_RED),
]
for i, (label, coef, se, color) in enumerate(pl_data):
    ax.bar(i, coef, yerr=se*1.96, color=color, capsize=6, width=0.55, edgecolor='white', linewidth=1, alpha=0.85)
    ax.text(i, coef+0.025, f'{coef:+.4f}', ha='center', fontsize=11, fontweight='bold', color=C_DARK)
ax.set_xticks(range(len(pl_data))); ax.set_xticklabels([d[0] for d in pl_data], fontsize=10, color=C_DARK)
add_zero_hline(ax)
ax.set_ylabel('DID 系数 (eff_apply_rd_10k)', fontsize=11, color=C_DARK)
ax.set_title('图 11  安慰剂检验结果图', fontsize=14, fontweight='bold', color=C_DARK)
ax.grid(True, alpha=0.25, axis='y', linestyle='--')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.annotate('假想时点也显著为正\n→ 效率差距在政策前\n   已存在收敛趋势', xy=(1.9,0.28),
            fontsize=9.5, color=C_RED, bbox=dict(boxstyle='round', facecolor='#FFF0F0', alpha=0.85, edgecolor=C_RED))
plt.tight_layout()
save_fig(fig, 'fig11_placebo')


In [ ]:
# ============================================================
# 图12: 研发费用加计扣除政策时间轴
# ============================================================
fig, ax = plt.subplots(figsize=(11.5, 3.8))
periods = [
    (2016.5,2018,'#F2F2F2','50%'), (2018,2021,'#DCE6F1','75%'),
    (2021,2023,'#B8CCE4','制造业 100%'), (2023,2025.5,'#95B3D7','全行业 100%'),
]
for start, end, color, label in periods:
    ax.axvspan(start, end, alpha=0.55, facecolor=color, edgecolor='#AAAAAA', linewidth=0.5)
    ax.text((start+end)/2, 0.62, label, ha='center', va='center', fontsize=10, fontweight='bold', color=C_DARK)
# DID窗口
ax.axvspan(2021, 2023, alpha=0.15, facecolor=C_RED)
ax.annotate('DID识别窗口\n制造业 vs 非制造业\n差异化激励', xy=(2022,0.32), fontsize=9.5, ha='center',
            fontweight='bold', color=C_RED, bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=C_RED, linewidth=1.5))
# 政策标注
policies = [
    (2018, '财税〔2018〕99号\n加计扣除75%', C_BLUE),
    (2021, '公告2021年第13号\n制造业100%', C_RED),
    (2022, '公告2022年第16号\n科小100%', C_ORANGE),
    (2023, '公告2023年第7号\n全行业100%', C_GREEN),
    (2023.8, '公告2023年第44号\nIC/母机120%', C_PURPLE),
]
for year, text, color in policies:
    ax.axvline(x=year, color=color, linestyle='-', linewidth=2, alpha=0.8)
    ax.text(year, 0.16, text, ha='center', fontsize=7.5, fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor=color))
ax.set_xlim(2016,2026); ax.set_ylim(0,1.05); ax.set_yticks([])
ax.set_xlabel('年份', fontsize=11, color=C_DARK)
ax.set_title('图 12  研发费用加计扣除政策时间轴', fontsize=14, fontweight='bold', color=C_DARK)
for spine in ['top','right','left']: ax.spines[spine].set_visible(False)
plt.tight_layout()
save_fig(fig, 'fig12_policy_timeline')

print('\n所有12张学术彩色图表已生成。')


## 14. 关键发现摘要

In [ ]:
print('=' * 70)
print('关键发现摘要')
print('=' * 70)

key_findings = {}
for dep, label in [
    ('ln_invention_apply', '发明专利申请'),
    ('ln_invention_grant', '发明专利授权'),
    ('ln_rd_expense_10k', '研发支出(万元对数)'),
    ('rd_intensity_01', '研发强度'),
    ('ln_rd_staff', '研发人员'),
    ('eff_apply_rd_10k', '效率(申请/支出)'),
    ('eff_grant_rd_10k', '效率(授权/支出)'),
    ('eff_apply_staff', '效率(申请/人员)'),
    ('eff_grant_staff', '效率(授权/人员)'),
]:
    matching = [r for r in decomp_all if r.get('dependent') == dep and 'error' not in r]
    if matching:
        r = matching[0]
        key_findings[label] = {
            'coef': r['manufacturing_post2021_coef'],
            'se': r['manufacturing_post2021_se'],
            'p': r['manufacturing_post2021_p'],
            'N': r.get('N', 0),
            'firms': r.get('firms', 0),
        }
        sig = fmt_sig(r['manufacturing_post2021_p'])
        print(f'  {label:25s}: {r["manufacturing_post2021_coef"]:+.4f} '
              f'({r["manufacturing_post2021_se"]:.4f}) p={r["manufacturing_post2021_p"]:.4f} {sig}  '
              f'N={r.get("N",0)}')

## 完成

### 输出文件清单

| 文件 | 路径 |
|------|------|
| 创新数量DID结果 | `outputs/focus_quantity_results.csv` |
| 研发投入调整DID结果 | `outputs/focus_rd_adjustment_results.csv` |
| 创新效率主结果 | `outputs/focus_efficiency_main_results.csv` |
| 效率拆解综合表 | `outputs/focus_efficiency_decomposition.csv` |
| 高研发基础异质性 | `outputs/focus_high_rd_heterogeneity.csv` |
| 制造业内部暴露异质性 | `outputs/focus_within_manufacturing_exposure.csv` |
| 所有制异质性 | `outputs/focus_ownership_heterogeneity.csv` |
| 政策暴露强度 | `outputs/focus_policy_exposure_results.csv` |
| 稳健性汇总 | `outputs/focus_robustness_summary.csv` |
| 事件研究 | `outputs/focus_event_study.csv` |
| 安慰剂检验 | `outputs/focus_placebo.csv` |
| 图表 (PNG+PDF) | `outputs/figures/fig*.png/pdf` |
| 数据审计报告 | `outputs/final_focus_data_audit.md` |

### 核心结论

1. 创新数量DID不显著: ln(发明申请) = −0.025, p = 0.3146
2. 研发投入DID显著为负: ln(研发支出) = −0.288, p < 0.001
3. 创新效率指标DID显著为正: 发明申请/研发支出效率 = +0.264, p < 0.001
4. 效率来源: DID(效率) ≈ DID(创新产出) − DID(研发投入) = +0.264
5. 事件研究和安慰剂检验提示因果解释需谨慎

**核心结论表述**: 研发费用加计扣除政策伴随研发投入调整和相对创新效率改善，
但现有结果不能支持严格因果意义上的政策效率提升。
效率指标是构造变量，其显著性在较大程度上受研发投入DID显著性的驱动，
能够说明单位投入产出指标的相对改善，但不能单独证明企业真实技术效率发生实质性提升。
